## Import libraries

In [1]:
from utils.dynamicRieszFunctions import estimateDynamicRiesz
from utils.dynamicRieszBradic import estimateBradic
import torch
import pandas as pd
import time

/Users/wisserutgers/anaconda3/lib/python3.10/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


# Generate data (look at different sizes of N)
tmax -> amount of simulations
N -> Sample size (per simulation)


In [2]:
torch.manual_seed(123)
torch.manual_seed(3)

In [3]:
lower = 0.05
upper = 0.95

# Define logistic function
def logistic(x):
    return torch.exp(x) / (1 + torch.exp(x))

# Define a truncated logistic function
def truncated_logistic(x):
    return lower + (upper - lower) * logistic(x)

treatment_probability_func = truncated_logistic

In [4]:
# Parameters
N = 1000
tmax = 1
d1 = 1 # Amount of covariates (keep fixed for these tests)
d2 = 1 # Amount of covariates (keep fixed for these tests)

# Coefficients
beta_pi1_0 = 0
beta_pi1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
beta_pi2_0_0 = 0
beta_pi2_0_S1 = torch.tensor([-0.5], dtype=torch.float32).reshape(-1,1) 
beta_pi2_0_S2 = torch.tensor([0.5], dtype=torch.float32).reshape(-1,1)
beta_pi2_1_0 = 0
beta_pi2_1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
beta_pi2_1_S2 = torch.tensor([-1], dtype=torch.float32).reshape(-1,1) 
beta_g0_0 = 1 
beta_g0_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
beta_g0_S2 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
beta_g1_0 = -1 
beta_g1_S1 = torch.tensor([1], dtype=torch.float32).reshape(-1,1) 
beta_g1_S2 = torch.tensor([-1], dtype=torch.float32).reshape(-1,1) 

# Generate data
S1_all = torch.randn(N * tmax, d1)
pi1_all = treatment_probability_func(beta_pi1_0 + S1_all @ beta_pi1_S1).reshape(-1,1)
A1_all = torch.bernoulli(pi1_all).int().reshape(-1,1)

delta1_all = torch.randn(N * tmax,1)
delta2_all = torch.randn(N * tmax, d2)

S2_all = S1_all + A1_all * (1 + delta1_all) + delta2_all

S2_1_all = S1_all + 1 + delta1_all + delta2_all
S2_0_all = S1_all + delta2_all

pi2_0_all = treatment_probability_func(beta_pi2_0_0 + S1_all @ beta_pi2_0_S1 + S2_0_all @ beta_pi2_0_S2)
pi2_1_all = treatment_probability_func(beta_pi2_1_0 + S1_all @ beta_pi2_1_S1 + S2_1_all @ beta_pi2_1_S2)
pi2_all = treatment_probability_func((1 - A1_all) * (beta_pi2_0_0 + S1_all @ beta_pi2_0_S1 + S2_0_all @ beta_pi2_0_S2) + 
                   A1_all * (beta_pi2_1_0 + S1_all @ beta_pi2_1_S1 + S2_1_all @ beta_pi2_1_S2))

A2_all = torch.bernoulli(pi2_all).int() 

g_all = ((A1_all + A2_all == 0).float() * (beta_g0_0 + S1_all @ beta_g0_S1 + S2_all @ beta_g0_S2) + 
         (A1_all * A2_all == 1).float() * (beta_g1_0 + S1_all @ beta_g1_S1 + S2_all @ beta_g1_S2))
zeta_all = torch.randn(N * tmax,1)
Y_all = g_all + zeta_all


##### True values:

In [5]:
mu2_1_all = beta_g1_0 + S1_all @ beta_g1_S1 + S2_1_all @ beta_g1_S2
mu2_0_all = beta_g0_0 + S1_all @ beta_g0_S1 + S2_0_all @ beta_g0_S2
mu1_1_all = beta_g1_0 + S1_all @ (beta_g1_S1 + beta_g1_S2) + beta_g1_S2.sum()
mu1_0_all = beta_g0_0 + S1_all @ (beta_g0_S1 + beta_g0_S2)

theta0 = beta_g0_0
theta1 = beta_g1_0 + beta_g1_S2.sum()
theta = theta1 - theta0
print(theta, theta1, theta0)

tensor(-3.) tensor(-2.) 1


## Estimation

In [6]:
folds = 2
time0 = time.time()

In [7]:
pred_theta = torch.zeros(tmax,5)
pred_sig = torch.zeros(tmax,5)

RR1 = torch.zeros(N,tmax,5)
RR2 = torch.zeros(N,tmax,5)

for t in range(0,tmax):

    # Get data for this iteration
    S1 = S1_all[(t) * N : (t+1) * N, :]
    S2 = S2_all[(t) * N : (t+1) * N, :]
    A1 = A1_all[(t) * N : (t+1) * N]
    A2 = A2_all[(t) * N : (t+1) * N]
    Y = Y_all[(t) * N : (t+1) * N]
    pi1 = pi1_all[(t) * N : (t+1) * N]
    pi2_0 = pi2_0_all[(t) * N : (t+1) * N]
    pi2_1 = pi2_1_all[(t) * N : (t+1) * N]
    mu2_1 = mu2_1_all[(t) * N : (t+1) * N]
    mu2_0 = mu2_0_all[(t) * N : (t+1) * N]
    mu1_1 = mu1_1_all[(t) * N : (t+1) * N]
    mu1_0 = mu1_0_all[(t) * N : (t+1) * N]

    X = torch.hstack((S1,S2))
    X_index = torch.tensor([S1.shape[1]-1,S1.shape[1]+S2.shape[1]-1])
    D = torch.hstack((A1,A2))

    # Set counterfactual of interest
    d = 1 * torch.ones(D.shape) 

    #---------------------------------------------------------------------------
    # Estimator 1: Oracle
    if d[0,0] == 1:
        gamma2_1 = A1 * A2 / (pi1 * pi2_1)
        gamma1_1 = A1 / pi1 
        theta_hat = (gamma2_1 * Y - (gamma1_1 - 1) * mu1_1   - (gamma2_1 - gamma1_1) * mu2_1)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
    elif d[0,0] == 0:
        gamma2_0 = (1 - A1) * (1 - A2) / ((1 - pi1  ) * (1 - pi2_0  ))
        gamma1_0 = (1 - A1) / (1 - pi1)
        theta_hat = (gamma2_0 * Y - (gamma1_0 - 1) * mu1_0   - (gamma2_0 - gamma1_0) * mu2_0)
        pred_theta[t,0] = torch.mean(theta_hat)
        pred_sig[t,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

    RR1[:,t,:1] = gamma1_1
    RR2[:,t,:1] = gamma2_1

    # #---------------------------------------------------------------------------
    # # Estimator 2: Bradic
    # bradic_result = estimateBradic(Y, X, D, X_index, folds)
    # if d[0,0] == 1:
    #     pred_theta[t,1] = bradic_result[7]
    #     pred_sig[t,1] = torch.sqrt(torch.tensor(bradic_result[10]))
    # elif d[0,0] == 0:
    #     pred_theta[t,1] = bradic_result[13]
    #     pred_sig[t,1] = torch.sqrt(torch.tensor(bradic_result[16]))

    #---------------------------------------------------------------------------
    # Estimator 3: LASSO 

    # point, sigma, RR1[:,t,2:3], RR2[:,t,2:3] = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "LASSO", method_f = "LASSO")
    # pred_theta[t,2] = point
    # pred_sig[t,2] = sigma

    #---------------------------------------------------------------------------
    # Estimator 4: RF 

    point, sigma, RR1[:,t,3:4], RR2[:,t,3:4] = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "RF", method_f = "RF")
    pred_theta[t,3] = point
    pred_sig[t,3] = sigma

    #---------------------------------------------------------------------------
    # Estimator 5: Net 

    point, sigma, RR1[:,t,4:5], RR2[:,t,4:5] = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "Net", method_f = "LASSO_CV")
    pred_theta[t,4] = point
    pred_sig[t,4] = sigma

    if t % 10 == 0:
        print("Time until iteration ",t, ": ", time.time() - time0)

print("Finished. Total time: ", time.time() - time0)


KeyboardInterrupt: 

## Compute statistics

### Right now: using theta1 (d = [1,1])

In [ ]:
true_value = theta1

In [ ]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO", "RF", "Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

In [ ]:
df = pd.DataFrame(data)
print(df)

   Method      Bias      RMSE  Coverage  Interval Length
0  Oracle -0.264289  0.264289       1.0         1.002107
1  Bradic  2.000000  2.000000       0.0         0.000000
2   LASSO  2.000000  2.000000       0.0         0.000000
3      RF -0.087994  0.087994       1.0         0.531575
4     Net -0.183130  0.183130       1.0         0.611690


In [ ]:
pred_theta

tensor([[-2.2643,  0.0000,  0.0000, -2.0880, -2.1831]])

## Compare riesz representers

In [ ]:
check_t = 0

# Compare all three:
pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)
# pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)

 
 # Currently: Estimation is terrible of NN

,0,1,2,3
0,3.645527,0.0,3.770008e+00,3.563572
1,0.000000,0.0,4.433280e-16,-0.790216
2,0.000000,0.0,1.776357e-15,1.464025
3,0.000000,0.0,1.222451e-14,-0.447690
4,3.871825,0.0,4.861887e+00,4.711696
5,11.782757,0.0,1.517303e+01,12.472541
6,0.000000,0.0,-1.001901e-15,-0.248987
7,0.000000,0.0,0.000000e+00,-0.351996
8,0.000000,0.0,-1.756125e-15,0.486403
9,0.000000,0.0,-1.776357e-15,-0.439910


## Compute statistics

### Right now: using theta1 (d = [1,1])

In [ ]:
true_value = theta1

In [ ]:
bias = torch.mean(pred_theta[:t+1,:] - true_value, 0)
rmse = torch.sqrt(torch.mean( (pred_theta[:t+1,:] - true_value) ** 2, 0))

ub = pred_theta[:t+1,:] + 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))
lb = pred_theta[:t+1,:] - 1.96 * pred_sig[:t+1,:] / torch.sqrt(torch.tensor(N))

coverage = torch.mean( ( (true_value >= lb) & (true_value <= ub) ).float() , 0 )
interval_length = torch.mean(ub - lb,0)

# Create a DataFrame
data = {
    "Method": ["Oracle", "Bradic", "LASSO", "RF", "Net"],
    "Bias": bias.ravel(),
    "RMSE": rmse.ravel(),
    "Coverage": coverage.ravel(),
    "Interval Length": interval_length.ravel()
}

In [ ]:
df = pd.DataFrame(data)
print(df)

   Method      Bias      RMSE  Coverage  Interval Length
0  Oracle -0.264289  0.264289       1.0         1.002107
1  Bradic  2.000000  2.000000       0.0         0.000000
2   LASSO  2.000000  2.000000       0.0         0.000000
3      RF -0.087994  0.087994       1.0         0.531575
4     Net -0.183130  0.183130       1.0         0.611690


In [ ]:
pred_theta

tensor([[-2.2643,  0.0000,  0.0000, -2.0880, -2.1831]])

## Compare riesz representers

In [ ]:
check_t = 0

# Compare all three:
pd.DataFrame(torch.hstack((RR2[:,check_t,0:1], RR2[:,check_t,2:3], RR2[:,check_t,3:4], RR2[:,check_t,4:5]))).head(50)
pd.DataFrame(torch.hstack((RR1[:,check_t,0:1], RR1[:,check_t,2:3], RR1[:,check_t,3:4], RR1[:,check_t,4:5]))).head(50)
 
 # Currently: Estimation is terrible of NN

,0,1,2,3
0,2.071408,0.0,2.797785,2.341284
1,1.723809,0.0,1.778039,1.708116
2,0.000000,0.0,0.000000,-0.027953
3,1.937658,0.0,1.875016,2.082977
4,1.552043,0.0,1.717891,1.517844
5,2.294035,0.0,2.877466,2.776307
6,1.247841,0.0,1.242881,1.077432
7,0.000000,0.0,0.000000,0.008993
8,2.047610,0.0,1.667094,2.080232
9,0.000000,0.0,0.000000,-0.057876


# Estimation with parallel comp

In [ ]:
# folds = 4

In [ ]:
# def process_iteration(t, N, S1_all, S2_all, A1_all, A2_all, Y_all, pi1_all, pi2_0_all, pi2_1_all, mu2_1_all, mu2_0_all, mu1_1_all, mu1_0_all, folds):

#     pred_theta = torch.zeros(1,5)
#     pred_sig = torch.zeros(1,5)

#     # Get data for this iteration
#     S1 = S1_all[(t) * N : (t+1) * N, :]
#     S2 = S2_all[(t) * N : (t+1) * N, :]
#     A1 = A1_all[(t) * N : (t+1) * N]
#     A2 = A2_all[(t) * N : (t+1) * N]
#     Y = Y_all[(t) * N : (t+1) * N]
#     pi1 = pi1_all[(t) * N : (t+1) * N]
#     pi2_0 = pi2_0_all[(t) * N : (t+1) * N]
#     pi2_1 = pi2_1_all[(t) * N : (t+1) * N]
#     mu2_1 = mu2_1_all[(t) * N : (t+1) * N]
#     mu2_0 = mu2_0_all[(t) * N : (t+1) * N]
#     mu1_1 = mu1_1_all[(t) * N : (t+1) * N]
#     mu1_0 = mu1_0_all[(t) * N : (t+1) * N]

#     X = torch.hstack((S1,S2))
#     X_index = torch.tensor([S1.shape[1]-1,S1.shape[1]+S2.shape[1]-1])
#     D = torch.hstack((A1,A2))

#     # Set counterfactual of interest
#     d = 1 * torch.ones(D.shape) 

#     #---------------------------------------------------------------------------
#     # Estimator 1: Oracle
#     if d[0,0] == 1:
#         gamma2_1 = A1 * A2 / (pi1 * pi2_1)
#         gamma1_1 = A1 / pi1 
#         theta_hat = (gamma2_1 * Y - (gamma1_1 - 1) * mu1_1   - (gamma2_1 - gamma1_1) * mu2_1)
#         pred_theta[0,0] = torch.mean(theta_hat)
#         pred_sig[0,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))
#     elif d[0,0] == 0:
#         gamma2_0 = (1 - A1) * (1 - A2) / ((1 - pi1  ) * (1 - pi2_0  ))
#         gamma1_0 = (1 - A1) / (1 - pi1)
#         theta_hat = (gamma2_0 * Y - (gamma1_0 - 1) * mu1_0   - (gamma2_0 - gamma1_0) * mu2_0)
#         pred_theta[0,0] = torch.mean(theta_hat)
#         pred_sig[0,0] = torch.sqrt(torch.mean((theta_hat - torch.mean(theta_hat) ) ** 2))

#     #---------------------------------------------------------------------------
#     #Estimator 2: Bradic
#     # bradic_result = estimateBradic(Y, X, D, X_index, folds)
#     # if d[0,0] == 1:
#     #     pred_theta[t,1] = bradic_result[7]
#     #     pred_sig[t,1] = bradic_result[10]
#     # elif d[0,0] == 0:
#     #     pred_theta[t,1] = bradic_result[13]
#     #     pred_sig[t,1] = bradic_result[16]

#     #---------------------------------------------------------------------------
#     # Estimator 3: LASSO 

#     # point, sigma, ub, lb = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "LASSO", method_f = "LASSO_CV", lasso_poly_degree = 1, c1 = "CV")
#     # pred_theta[0,2] = point
#     # pred_sig[0,2] = sigma

#     #---------------------------------------------------------------------------
#     # Estimator 4: RF 

#     point, sigma, ub, lb = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "RF", method_f = "RF", l2 = 0)
#     pred_theta[0,3] = point
#     pred_sig[0,3] = sigma

#     # # # #---------------------------------------------------------------------------
#     # # # # Estimator 5: Net 

#     # point, sigma, ub, lb = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "LASSO", method_f = "Net")
#     # pred_theta[0,4] = point
#     # pred_sig[0,4] = sigma

#     point, sigma, ub, lb = estimateDynamicRiesz(Y, X, D, d, X_index, folds, method_a = "RF", method_f = "Net")
#     pred_theta[0,2] = point
#     pred_sig[0,2] = sigma
    
#     return pred_theta[0,:], pred_sig[0,:]

In [ ]:
# results = []
# time0 = time.time()

# with ThreadPoolExecutor() as executor:  # Use ProcessPoolExecutor for CPU-bound tasks
#     futures = [executor.submit(process_iteration, t, N, S1_all, S2_all, A1_all, A2_all, Y_all, pi1_all, pi2_0_all, pi2_1_all, mu2_1_all, mu2_0_all, mu1_1_all, mu1_0_all, folds) for t in range(tmax)]
#     for future in as_completed(futures):
#         results.append(future.result())


# print(time.time() - time0)

In [ ]:
# pred_theta_list = [res[0] for res in results]  # Extract all pred_theta vectors
# pred_sig_list = [res[1] for res in results]    # Extract all pred_sig vectors

# # Combine into matrices
# pred_theta = torch.stack(pred_theta_list)  # Combine into a single matrix
# pred_sig = torch.stack(pred_sig_list)      # Combine into a single matrix
